In [1]:
# ! pip install datasets trl
! pip install datasets git+https://github.com/huggingface/trl.git@v0.25.0

  Cloning https://github.com/huggingface/trl.git (to revision v0.25.0) to /tmp/pip-req-build-ao_s_3qy
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/trl.git /tmp/pip-req-build-ao_s_3qy
  Running command git checkout -q 9692b1688e70fbceb89d244a9bb4a73d28879c24
  Resolved https://github.com/huggingface/trl.git to commit 9692b1688e70fbceb89d244a9bb4a73d28879c24
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trl: filename=trl-0.25.0-py3-none-any.whl size=462827 sha256=4c4ca172b5b32ad1034f79db0cd4625ebd35d906d8751903d41636382a01ae72
  Stored in directory: /tmp/pip-ephem-wheel-cache-34u9jgye/wheels/a8/64/6e/212232d1b6e47b0b45974b2b95726ae29d691b9878ee08c8ee
Successfully built trl


# SFT

In this section, we will instruction-tune a small model, i.e. we will make sure it follows a conversation-like pattern. We will do that using [TRL - Transformer Reinforcement Learning](https://github.com/huggingface/trl/blob/main/README.md) library.

Check out the project's website. Which other common alignment-related features are supported?

In [2]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/trl/models/utils.py:117: FutureWarning: The `setup_chat_format` function is deprecated and will be removed in version 0.26.0. Please use `clone_chat_template` instead.
  warnings.warn(


Notice we have used `setup_chat_format` function. Read about chat formats [here](https://huggingface.co/docs/transformers/main/en/chat_templating).



Now that we have loaded a base model, let's try using it. How does it perform? Why is that?

In [3]:
# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
# TODO: Generate model's response
tokenized_chat = tokenizer.encode(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(tokenized_chat, max_new_tokens=128)
print(tokenizer.decode(outputs[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|im_start|>user
Write a haiku about programming<|im_end|>
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a


Let's load a proper dataset

In [4]:
# Load a sample dataset
from datasets import load_dataset

# TODO: load everyday-conversations from the HuggingFaceTB/smoltalk dataset
ds = load_dataset("HuggingFaceTB/smoltalk", "everyday-conversations")

README.md: 0.00B [00:00, ?B/s]

data/everyday-conversations/train-00000-(…):   0%|          | 0.00/946k [00:00<?, ?B/s]

data/everyday-conversations/test-00000-o(…):   0%|          | 0.00/52.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

Now that we have a model and a dataset, we can simply create a config and run the SFTTrainer. Training should take around 5 minutes.

In [9]:
# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    eval_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=False,
    report_to="tensorboard",
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    # tokenizer=tokenizer,
    processing_class=tokenizer, # STUDENT"S change
    eval_dataset=ds["test"],
)

In [10]:
# Train the model
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.751000,1.074794,0.943459,38455.000000,0.723185
100,0.812200,1.069688,0.972690,77423.000000,0.725981
150,0.803400,1.070064,0.941502,115538.000000,0.726703
200,0.866700,1.051154,0.947375,154355.000000,0.732776
250,1.002400,1.029768,1.015637,193006.000000,0.733684
300,0.997500,1.021797,1.023804,232247.000000,0.735285
350,0.969100,1.016757,1.034993,270666.000000,0.735190
400,0.974200,1.014574,0.992440,309790.000000,0.737398
450,0.988300,1.004338,1.015060,348437.000000,0.737661
500,1.042800,0.995468,1.007069,388174.000000,0.737471


TrainOutput(global_step=1000, training_loss=0.8495012173652648, metrics={'train_runtime': 1146.2305, 'train_samples_per_second': 3.49, 'train_steps_per_second': 0.872, 'total_flos': 587568496250880.0, 'train_loss': 0.8495012173652648, 'epoch': 1.7699115044247788})

We have fine-tuned a model. It should follow the conversation format now (to some extent - it is a small model fine-tuned for a very short time).

Play with the model a bit.

In [11]:
# Test the fine-tuned model on the same prompt

# Let's test the model after SFT
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
print("After Training")

# TODO: use the fine-tuned to model generate a response, just like with the base example.
print("FIRST")
tokenized_chat = tokenizer.encode(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(tokenized_chat, max_new_tokens=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
# why there is no add_generation_prompt=True?
print("SECOND:")
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
tokenized_chat = tokenizer.encode(formatted_prompt, return_tensors="pt").to(device)
# outputs = model.generate(tokenized_chat, max_new_tokens=128)

outputs = model.generate(
    tokenized_chat,
    max_new_tokens=80,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)

print(tokenizer.decode(outputs[0]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


After Training
FIRST
user
Write a haiku about programming

SECOND:
<|im_start|>user
Write a haiku about programming<|im_end|>
<|im_start|>assistant
I'm looking for some programming haikus. Do you know of any?<|im_end|>


Optionally, we can save the model.

In [12]:
# TODO: save the model to a local path and download it or upload it to huggingface.
# Local save:
save_path = './smolLM2'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

('./smolLM2/tokenizer_config.json',
 './smolLM2/special_tokens_map.json',
 './smolLM2/chat_template.jinja',
 './smolLM2/vocab.json',
 './smolLM2/merges.txt',
 './smolLM2/added_tokens.json',
 './smolLM2/tokenizer.json')

In [ ]:
# HuggingFace:
# ! pip install -U huggingface_hub
!pip install -U "huggingface_hub<1.0"
! huggingface-cli login

In [ ]:
repo_name = "smollm2-sft-haiku"
model.push_to_hub(repo_name, private=True)
tokenizer.push_to_hub(repo_name, private=True)

In [ ]:
# Hugging Face - optional download:
from transformers import AutoModelForCausalLM, AutoTokenizer

repo_id = "KamilloP/smollm2-sft-haiku"

tokenizer2 = AutoTokenizer.from_pretrained(repo_id)
model2 = AutoModelForCausalLM.from_pretrained(repo_id)


# Evaluating reward model
In this task, we will be evaluating how closely a reward model's preferences follow preferences represented in a preference dataset.

What is a reward model? Which alignment algorithm uses a reward model?

In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import torch

dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

We have loaded the dataset. Inspect it.

Each row contains a pair - a rejected and accepted answer. Your task is to compute the accuracy of the model, i.e. how often the accepted answer is ranked higher than the rejected answer by the reward model.

To see how to use the model, look up its card on HuggingFace: [link](https://huggingface.co/OpenAssistant/reward-model-deberta-v3-large-v2).

In [14]:
def get_reward(model, tokenizer, question, answer):
    # TODO begin
    inputs = tokenizer(question, answer, return_tensors="pt").to(device)
    return model(**inputs).logits[0].cpu().detach()
    # TODO end

model_name = "OpenAssistant/reward-model-deberta-v3-large-v2"
model, tokenizer = AutoModelForSequenceClassification.from_pretrained(model_name).to(device), AutoTokenizer.from_pretrained(model_name)

correct = []
with torch.no_grad():
    for row in dataset.shuffle().select(range(100)):
        # print(row)
        question, rejected, chosen = row["chosen"][0]['content'], row["rejected"][1]['content'], row["chosen"][1]['content']
        chosen_score = get_reward(model, tokenizer, question, chosen)
        rejected_score = get_reward(model, tokenizer, question, rejected)
        correct.append((chosen_score > rejected_score).item())

print("Accuracy: ", sum(correct) / len(correct))


config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Accuracy:  0.68


# Is an aligned model aligned with the human preferences?

In this task, we will load an instruct-tuned (and DPO'd) LLM and evaluate if it rates positive examples from a preference dataset as more probable than the rejected examples.

To format the input properly, re-use chat formatting from the SFT task. To figure out which response is preferred by the model, use the normalized sum of log-probabilities of the continuation tokens (we talked about that in the lm-eval-harness lab - [link](https://blog.eleuther.ai/multiple-choice-normalization/). You can treat the tokens between the last `<|im_start|>` token and the last `<|im_end|>` token as the continuation.



In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import setup_chat_format
import torch.nn.functional as F

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to('cuda')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)


def get_logprobs(model, tokenizer, question, answer):
    # TODO begin
    messages = [{"role": "user", "content": question}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    full_text = prompt_text + "\n" + answer
    inputs = tokenizer(full_text, return_tensors="pt").to('cuda')

    prompt_len = len(tokenizer(prompt_text)["input_ids"])
    answer_ids = inputs["input_ids"][0, prompt_len:]

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    logprobs = F.log_softmax(logits[0, :-1], dim=-1)
    selected_logprobs = logprobs[torch.arange(len(answer_ids)), answer_ids]

    normalized_logprob = selected_logprobs / (1+torch.arange(len(answer_ids)).to(device))
    return normalized_logprob.sum()
    # TODO end


correct = []
with torch.no_grad():
    for row in dataset.shuffle().select(range(100)): # you can increase the number of samples
        # TODO: get normalized (by the number of tokens) sum of logprobs for both chosen and rejected continuations
        question, rejected, chosen = row["chosen"][0]['content'], row["rejected"][1]['content'], row["chosen"][1]['content']
        chosen_logprob = get_logprobs(model, tokenizer, question, chosen)
        rejected_logprob = get_logprobs(model, tokenizer, question, rejected)
        correct.append((chosen_logprob > rejected_logprob).item())
print("RLHF model's accuracy:", sum(correct) / len(correct))


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

RLHF model's accuracy: 0.48
